# Flood Risk Prediction & Automated Early-Warning Alert System
### Regression Analysis Assignment — Data Science & Analytics (5th Sem)

**Domain:** Environmental Studies / Disaster Management
**Task type:** Regression (predicting a continuous Flood Probability / Risk Score)
**Deployment concept:** Instead of a "check-it-yourself" app, the trained model powers a backend
service that scores incoming regional data and automatically pushes SMS/alert notifications to
registered users the moment predicted risk crosses a danger threshold — mirroring real early-warning
systems (e.g. IMD heavy-rain SMS alerts).


## 1. Problem Definition

**Background:** Floods remain one of the most frequent and destructive natural disasters
worldwide (e.g. the 2026 Nepal floods). Most existing "flood prediction apps" assume users will
proactively check a dashboard — in practice, people don't. This project instead treats prediction
as the *trigger* for an automated push-alert pipeline.

**Objective:** Build a regression model that predicts a continuous **Flood Probability / Risk
Score (0–1)** for a region from environmental & human factors, validate it rigorously, and wire it
into a deployable alert-broadcast application.

**Target variable:** `FloodProbability` (continuous, 0–1)

**Prediction task:** Multivariate regression.


## 2. Setup & Data Loading

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import joblib, os, warnings
warnings.filterwarnings("ignore")
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)


In [ ]:
# Robust loader — works whether the CSV sits at root, in /content, or in a data/ subfolder
# (mirrors the path issue you hit on the concrete-strength project)
CANDIDATE_PATHS = [
    "train.csv",
    "FloodPrediction.csv",
    "data/train.csv",
    "/content/train.csv",
    "/content/drive/MyDrive/train.csv",
]

df = None
for p in CANDIDATE_PATHS:
    if os.path.exists(p):
        df = pd.read_csv(p)
        print(f"Loaded dataset from: {p}")
        break

if df is None:
    raise FileNotFoundError(
        "Couldn't find the dataset. Upload train.csv (Kaggle 'Flood Prediction Factors' / "
        "Playground Series S4E5) to the notebook's working directory, or edit CANDIDATE_PATHS above."
    )

print(df.shape)
df.head()


In [ ]:
# This dataset (Kaggle Playground S4E5) has ~1.1M rows — great for the "500+ observations"
# requirement, but Random Forest/XGBoost on the full set is slow (and can time out) on Colab's
# free tier. We sample down to a fast-but-still-robust size. Bump SAMPLE_SIZE up if you have
# Colab Pro / more RAM, or set it to None to use the full dataset.
SAMPLE_SIZE = 50000

if SAMPLE_SIZE is not None and len(df) > SAMPLE_SIZE:
    df = df.sample(n=SAMPLE_SIZE, random_state=42).reset_index(drop=True)
    print(f"Sampled down to {len(df)} rows for faster training (report this in your Dataset Description).")
else:
    print(f"Using full dataset: {len(df)} rows.")


## 3. Data Understanding

**Dataset source:** Kaggle — "Regression with a Flood Prediction Dataset" (Playground Series S4E5),
based on the original "Flood Prediction Factors" dataset.
**Target variable:** `FloodProbability` — continuous risk score (0–1)
**Predictor variables:** ~20 environmental & socio-economic factors (monsoon intensity, deforestation,
urbanization, drainage systems, dams quality, siltation, climate change score, population score,
infrastructure quality, political/planning factors, etc.)


In [ ]:
df.info()
print("\nMissing values per column:\n", df.isnull().sum().sum(), "total missing cells")
df.describe().T


In [ ]:
# Drop id column if present (not predictive)
if "id" in df.columns:
    df = df.drop(columns=["id"])

target_col = "FloodProbability"
assert target_col in df.columns, f"Expected target column '{target_col}' not found — check dataset columns: {list(df.columns)}"


In [ ]:
# Target distribution
fig, ax = plt.subplots(1, 2, figsize=(14, 4))
sns.histplot(df[target_col], kde=True, ax=ax[0])
ax[0].set_title("Distribution of FloodProbability")
sns.boxplot(x=df[target_col], ax=ax[1])
ax[1].set_title("Boxplot — FloodProbability")
plt.tight_layout()
plt.show()


In [ ]:
# Correlation heatmap — which factors move with flood risk?
plt.figure(figsize=(14, 11))
corr = df.corr(numeric_only=True)
sns.heatmap(corr, cmap="coolwarm", center=0, annot=False)
plt.title("Feature Correlation Heatmap")
plt.show()

corr[target_col].sort_values(ascending=False)


In [ ]:
# Top correlated features with target — bar chart
top_corr = corr[target_col].drop(target_col).sort_values(key=abs, ascending=False).head(10)
plt.figure(figsize=(9, 5))
sns.barplot(x=top_corr.values, y=top_corr.index, palette="viridis")
plt.title("Top 10 Features Correlated with FloodProbability")
plt.xlabel("Correlation coefficient")
plt.show()


## 4. Data Pre-processing

- Missing value check (imputation if needed)
- Outlier detection via IQR
- Feature scaling (StandardScaler) — needed for Linear/Ridge/Lasso baselines
- Train/test split (80/20) with a fixed random_state for reproducibility


In [ ]:
# Missing value treatment (median impute, if any)
if df.isnull().sum().sum() > 0:
    df = df.fillna(df.median(numeric_only=True))
    print("Missing values imputed with column medians.")
else:
    print("No missing values found.")

# Outlier detection (IQR method) — report only, not auto-dropped (environmental data legitimately varies)
Q1 = df.quantile(0.25, numeric_only=True)
Q3 = df.quantile(0.75, numeric_only=True)
IQR = Q3 - Q1
outlier_counts = ((df.select_dtypes(include=np.number) < (Q1 - 1.5 * IQR)) |
                   (df.select_dtypes(include=np.number) > (Q3 + 1.5 * IQR))).sum()
print("Outlier counts per column (IQR method):\n", outlier_counts[outlier_counts > 0])


In [ ]:
X = df.drop(columns=[target_col])
y = df[target_col]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Train shape:", X_train.shape, " Test shape:", X_test.shape)


## 5. Model Development

**Baseline:** Multiple Linear Regression
**Candidate models:** Ridge, Lasso, Random Forest Regressor, XGBoost Regressor (if installed)

We justify Random Forest / XGBoost as primary candidates because flood risk factors interact
non-linearly (e.g. deforestation only matters when drainage is also poor) — tree ensembles capture
these interactions far better than a linear baseline, which we still report for comparison as
required by the brief.


In [ ]:
try:
    from xgboost import XGBRegressor
    HAS_XGB = True
except ImportError:
    HAS_XGB = False
    print("xgboost not installed — run: !pip install xgboost")

models = {
    "Linear Regression (baseline)": LinearRegression(),
    "Ridge Regression": Ridge(alpha=1.0),
    "Lasso Regression": Lasso(alpha=0.001),
    "Random Forest Regressor": RandomForestRegressor(n_estimators=300, max_depth=12, random_state=42, n_jobs=-1),
}
if HAS_XGB:
    models["XGBoost Regressor"] = XGBRegressor(n_estimators=400, max_depth=6, learning_rate=0.05,
                                                 subsample=0.8, colsample_bytree=0.8, random_state=42)


In [ ]:
results = []

for name, model in models.items():
    use_scaled = "Regression" in name and "Forest" not in name and "Boost" not in name
    Xtr = X_train_scaled if use_scaled else X_train
    Xte = X_test_scaled if use_scaled else X_test

    model.fit(Xtr, y_train)
    preds = model.predict(Xte)

    mae = mean_absolute_error(y_test, preds)
    mse = mean_squared_error(y_test, preds)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_test, preds)

    # 5-fold cross validation (R²)
    cv_scores = cross_val_score(model, Xtr, y_train, cv=KFold(5, shuffle=True, random_state=42), scoring="r2")

    results.append({
        "Model": name, "MAE": mae, "MSE": mse, "RMSE": rmse, "R2": r2,
        "CV_R2_mean": cv_scores.mean(), "CV_R2_std": cv_scores.std()
    })
    print(f"{name:32s} | R2={r2:.4f} | RMSE={rmse:.4f} | CV_R2={cv_scores.mean():.4f}±{cv_scores.std():.4f}")

results_df = pd.DataFrame(results).sort_values("R2", ascending=False).reset_index(drop=True)
results_df


In [ ]:
# Model comparison chart
plt.figure(figsize=(9, 5))
sns.barplot(data=results_df, x="R2", y="Model", palette="mako")
plt.title("Model Comparison — R² Score (higher is better)")
plt.xlabel("R² score")
plt.tight_layout()
plt.show()


## 6. Model Validation

Best model is selected by highest cross-validated R². We inspect residuals to confirm the model
isn't systematically over/under-predicting risk at any range — critical here since under-predicting
high-risk regions is the costly error for an alert system.


In [ ]:
best_model_name = results_df.iloc[0]["Model"]
best_model = models[best_model_name]
print("Best model:", best_model_name)

use_scaled = "Regression" in best_model_name and "Forest" not in best_model_name and "Boost" not in best_model_name
Xte_best = X_test_scaled if use_scaled else X_test
best_preds = best_model.predict(Xte_best)

residuals = y_test - best_preds
fig, ax = plt.subplots(1, 2, figsize=(14, 5))
ax[0].scatter(best_preds, residuals, alpha=0.3)
ax[0].axhline(0, color="red", linestyle="--")
ax[0].set_xlabel("Predicted FloodProbability")
ax[0].set_ylabel("Residual")
ax[0].set_title("Residual Plot")

ax[1].scatter(y_test, best_preds, alpha=0.3)
ax[1].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], "r--")
ax[1].set_xlabel("Actual FloodProbability")
ax[1].set_ylabel("Predicted FloodProbability")
ax[1].set_title("Predicted vs Actual")
plt.tight_layout()
plt.show()


## 7. Model Interpretation

Feature importance (tree-based models) or coefficients (linear models) tell us which factors
drive flood risk the most — this becomes the "why" section of the presentation and the basis for
which live inputs the alert app actually needs to collect/monitor.


In [ ]:
if hasattr(best_model, "feature_importances_"):
    importances = pd.Series(best_model.feature_importances_, index=X.columns).sort_values(ascending=False)
elif hasattr(best_model, "coef_"):
    importances = pd.Series(np.abs(best_model.coef_), index=X.columns).sort_values(ascending=False)
else:
    importances = None

if importances is not None:
    plt.figure(figsize=(9, 8))
    sns.barplot(x=importances.head(15).values, y=importances.head(15).index, palette="rocket")
    plt.title(f"Top 15 Important Predictors — {best_model_name}")
    plt.xlabel("Importance")
    plt.tight_layout()
    plt.show()
    importances.head(15)


In [ ]:
# Optional: SHAP explainability (uncomment if shap is installed — pip install shap)
# import shap
# explainer = shap.Explainer(best_model, X_train)
# shap_values = explainer(X_test.iloc[:200])
# shap.summary_plot(shap_values, X_test.iloc[:200])


## 8. Export Model for Deployment

Saves the trained model + scaler + feature list so the Flask alert app (built separately) can load
them directly — no retraining needed at deploy time.


In [ ]:
os.makedirs("model_artifacts", exist_ok=True)
joblib.dump(best_model, "model_artifacts/flood_model.pkl")
joblib.dump(scaler, "model_artifacts/scaler.pkl")
joblib.dump(list(X.columns), "model_artifacts/feature_columns.pkl")
joblib.dump(use_scaled, "model_artifacts/uses_scaling.pkl")

print("Saved: model_artifacts/flood_model.pkl, scaler.pkl, feature_columns.pkl")
print("Best model:", best_model_name, "| Test R2:", results_df.iloc[0]['R2'])


## 9. Conclusion, Limitations & Future Scope

**Conclusion:** The chosen model predicts regional flood probability with strong R² / low RMSE,
and — critically — is deployed as the engine behind a *push-alert* application rather than a
passive lookup tool, directly addressing the real-world adoption gap in most flood-prediction
projects.

**Limitations:** Model quality depends on data recency/granularity for a specific region; no
real-time satellite/rainfall feed is wired in for this coursework version; alert delivery in the
demo uses a Twilio trial account (production would need a paid SMS gateway + verified sender IDs
and integration with a live meteorological API).

**Future scope:** Live weather API integration (OpenWeatherMap/IMD), per-ward granularity,
multilingual SMS templates, and a severity-tiered alert cadence (advisory → warning → evacuation).
